### READ DATA FROM BRONZE TABLE


In [0]:
df = spark.table('workspace.bronze_schema.prod_info_delta')
df.display()

### NORMALIZE/RENAME THE COLUMN NAMES


In [0]:
#Create a dictionary to hold the new and old column names
col_dict = {'prd_id':'production_id', 'prd_key':'production_key', 'prd_nm':'production_name', 'prd_cost':'production_cost', 'prd_line':'production_line', 'prd_start_dt':'production_end_date', 'prd_end_dt':'production_start_date'}
#Rename the columns in the DataFrame
for old_col, new_col in col_dict.items():
  df = df.withColumnRenamed(old_col, new_col)
#Display the DataFrame
df.display()


In [0]:
%sql
select prd_nm from workspace.bronze_schema.prod_info_delta

### FILL NA VALUES

In [0]:
#df.fillna('Not_available').display()

df.fillna('Not_available', subset=['production_line']).display()

In [0]:
from pyspark.sql.functions import col

df = df.fillna({'production_cost': 0})


In [0]:
df = df.fillna('Not_available', subset=['production_line'])
df.display()

### TRIM CERTAIN COLUMNS FOR SPACES

In [0]:
from pyspark.sql.functions import trim
df = df.withColumn('production_line',trim(col('production_line')))
df.display()

In [0]:
df = df.fillna({'production_end_date': '1900-01-01'})
df.display()

### RENAME VALUES IN PRODUCTION_LINE COLUMN

In [0]:
df = df.withColumn('production_line', when(col('production_line') == 'S', 'Other_Sales')\
    .when(col('production_line') == 'R', 'Road')\
    .when(col('production_line') == 'T', 'Touring')\
    .when(col('production_line') == 'M', 'Mountain')\
    .otherwise('Not_available'))
df.display()

In [0]:
df.printSchema()


In [0]:
df.withColumn('production_end_date', trim(col('production_end_date'))).display()

In [0]:
from pyspark.sql.functions import coalesce, col, lit

df = df.withColumn(
    'production_start_date',
    coalesce(
        col('production_start_date'),
        lit('2000-01-01').cast('date')
    )
)

df.display()


In [0]:
from pyspark.sql.functions import datediff
df.withColumn('Production_date_diff',datediff(col('production_end_date'),col('production_start_date'))).display()

### WRITE TRANSFORMED DATA INTO SILVER LAYER

In [0]:
df.write.mode('overwrite').saveAsTable('workspace.silver_schema.production_info')